# Lab 15: Q-Learning on FrozenLake

**Module 15** | Read `notes/15-rl-mdps-qlearning.pdf` before running this notebook.

Tabular Q-learning with epsilon-greedy exploration on FrozenLake-v1 and render a solved episode.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Tabular Q-learning on FrozenLake

**Reinforcement learning** agents learn by trial and error. An **MDP** (Markov Decision Process) is the formal setup: states, actions, rewards, and transition rules. FrozenLake-v1 is a small MDP on a 4x4 grid.

**Q-learning** learns how good each action is in each state without building a model of the environment. The learned values live in a **Q-table**: a 16-by-4 table (16 states, 4 actions) storing expected future reward.

**Exploration vs exploitation:** An agent that only picks the best-known action (*exploitation*) may never discover the goal. An agent that acts randomly (*exploration*) wanders wastefully. **Epsilon-greedy** balances both: with probability `epsilon` pick a random action, otherwise pick the best action according to the Q-table.

This lab trains with `is_slippery=False` (deterministic moves), prints learning progress, shows the learned policy as arrows, and replays one greedy episode.


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **MDP** | Markov Decision Process. A framework with states, actions, rewards, and transition rules. The next state depends only on the current state and action (Markov property). |
| **Q-learning** | A tabular RL algorithm that updates action-value estimates from experience using the Bellman equation. No environment model required. |
| **Q-table** | A lookup table Q(state, action) storing expected cumulative reward for taking an action in a state and then behaving optimally. |
| **Epsilon-greedy** | Exploration strategy: with probability epsilon choose a random action; otherwise choose argmax Q. Epsilon balances trying new moves vs using what already works. |
| **Exploration** | Trying actions you are unsure about to discover better strategies (e.g., finding the path to the goal). |
| **Exploitation** | Choosing the action that currently looks best according to the Q-table. |
| **Discount factor (gamma)** | How much future rewards matter compared to immediate rewards. gamma near 1 means the agent cares about long-term payoff. |

Refer back to this table whenever a term appears in code or output.


### Step 1: Environment and hyperparameters

**What this section does:** Creates FrozenLake-v1, allocates a zero Q-table, and prints hyperparameters (learning rate, discount, epsilon, episode count).

**Why we do it:** FrozenLake has 16 discrete states and 4 actions (left, down, right, up). Printing settings up front makes runs reproducible and easier to tune later.


**What to look for in the output**

- States: 16, Actions: 4.
- Learning rate (alpha): 0.8, gamma: 0.99, epsilon: 0.1.
- Episodes: 10,000.
- Action mapping printed (LEFT, DOWN, RIGHT, UP).


In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("FrozenLake-v1", is_slippery=False)
nS = env.observation_space.n  # 16 grid cells
nA = env.action_space.n       # 4 directions
Q = np.zeros((nS, nA))        # Q-table: all values start at zero (no prior knowledge).

ALPHA = 0.8       # Learning rate: how fast Q-values move toward new targets.
GAMMA = 0.99      # Discount: weight on future rewards.
EPSILON = 0.1     # Exploration rate: 10% random actions, 90% greedy.
N_EPISODES = 10_000
REPORT_EVERY = 2000

ACTION_NAMES = {0: "LEFT", 1: "DOWN", 2: "RIGHT", 3: "UP"}
ACTION_SYMBOLS = {0: "<-", 1: "v", 2: "->", 3: "^"}

print("Q-learning hyperparameters:")
print(f"  States (nS):        {nS}")
print(f"  Actions (nA):       {nA}")
print(f"  Learning rate:      {ALPHA}")
print(f"  Discount (gamma):   {GAMMA}")
print(f"  Exploration (eps):  {EPSILON}")
print(f"  Episodes:           {N_EPISODES:,}")
print(f"  Progress report:    every {REPORT_EVERY} episodes")
print()
print("Action mapping:", ACTION_NAMES)


### Step 2: Q-learning training loop

**What this section does:** Runs 10,000 episodes. Each step picks an epsilon-greedy action, observes reward and next state, and applies the Q-learning update. Success means reaching the goal (reward 1.0).

**Why we do it:** The Bellman update nudges Q(s, a) toward `reward + gamma * max Q(next_state)`. Over many episodes the table converges to useful values on this small map. Progress reports every 2,000 episodes show success rate climbing.


**What to look for in the output**

- Progress lines every 2000 episodes with success rate percentage.
- Success rate should rise over time on the deterministic map (often toward 100%).
- Final success rate (last 1000 episodes) printed at the end.


In [ ]:
rewards_per_episode: list[float] = []

for episode in range(N_EPISODES):
    state, _ = env.reset()
    done = False
    total_reward = 0.0

    while not done:
        # Epsilon-greedy: explore randomly sometimes, exploit best Q-value otherwise.
        if np.random.random() < EPSILON:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q[state]))

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward

        # Q-learning update (tabular Bellman backup).
        best_next = np.max(Q[next_state])
        Q[state, action] += ALPHA * (reward + GAMMA * best_next - Q[state, action])
        state = next_state

    rewards_per_episode.append(total_reward)

    if (episode + 1) % REPORT_EVERY == 0:
        window = rewards_per_episode[-REPORT_EVERY:]
        success_rate = 100.0 * np.mean(window)
        successes = int(np.sum(window))
        print(
            f"Episode {episode + 1:>5}/{N_EPISODES}: "
            f"success rate last {REPORT_EVERY} = {success_rate:5.1f}% "
            f"({successes}/{REPORT_EVERY} goals reached)"
        )

final_window = rewards_per_episode[-1000:]
final_success = 100.0 * np.mean(final_window)
print()
print(f"Final success rate (last 1000 episodes): {final_success:.1f}%")


### Step 3: Q-table statistics

**What this section does:** Summarizes min, max, mean, and non-zero entry count in the Q-table, then lists the five states with highest best-action Q-values.

**Why we do it:** States near the goal and along successful paths typically carry the highest values. Many entries stay at zero on states the agent rarely visits.


**What to look for in the output**

- Q-table shape 16 x 4.
- Max Q-value noticeably above zero after training.
- Top states list shows which grid positions the agent values most.


In [ ]:
print("Q-table statistics:")
print(f"  Shape:           {Q.shape}")
print(f"  Min value:       {Q.min():.4f}")
print(f"  Max value:       {Q.max():.4f}")
print(f"  Mean value:      {Q.mean():.4f}")
print(f"  Non-zero entries: {np.count_nonzero(Q)} / {Q.size}")
print()

best_state_actions = []
for s in range(nS):
    best_a = int(np.argmax(Q[s]))
    best_state_actions.append((s, best_a, Q[s, best_a]))

top_states = sorted(best_state_actions, key=lambda x: x[2], reverse=True)[:5]
print("Top 5 states by best-action Q-value:")
for s, a, val in top_states:
    print(f"  state {s:2d}: best={ACTION_NAMES[a]:<6} Q={val:.4f}")


### Step 4: Learned policy grid

**What this section does:** Builds the greedy policy (`argmax` over actions per state) and displays it as arrows on the 4x4 map. S=start, G=goal, H=hole.

**Why we do it:** A policy is easier to read than raw Q-numbers. Arrows should point toward the goal while avoiding holes.


**What to look for in the output**

- 4x4 grid with pipes between cells.
- Start (S) and goal (G) unchanged; holes (H) show letter H.
- Other cells show direction arrows (v, ->, etc.) pointing toward the goal path.


In [ ]:
MAP_LAYOUT = [
    "SFFF",
    "FHFH",
    "FFFH",
    "HFFG",
]

policy = np.argmax(Q, axis=1)
print("Learned policy (4x4 grid, row-major):")
print("-" * 17)
for row_idx, row in enumerate(MAP_LAYOUT):
    cells = []
    for col_idx, ch in enumerate(row):
        state = row_idx * 4 + col_idx
        if ch in "SHG":
            cells.append(f" {ch} ")
        else:
            cells.append(f" {ACTION_SYMBOLS[policy[state]]} ")
    print("|".join(cells))
print("-" * 17)
print("S=start  G=goal  H=hole  arrows=greedy action")


### Step 5: Render one solved episode

**What this section does:** Replays one episode using only the greedy policy (no exploration). Prints the map after each step with action names and cumulative reward.

**Why we do it:** Confirms the learned policy actually reaches the goal when exploitation alone drives every action.


**What to look for in the output**

- ASCII map printed at start and after each step.
- Actions listed as LEFT, DOWN, RIGHT, or UP.
- Cumulative reward reaches 1.0 if the goal is reached.
- `Reached the goal in N steps` message on success.


In [ ]:
render_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")
state, _ = render_env.reset()
cumulative = 0.0

print("Greedy episode replay:")
print(render_env.render())
print(f"Step 0: start state={state}, cumulative reward={cumulative:.1f}")
print()

steps = 0
done = False

while not done and steps < 20:
    action = int(np.argmax(Q[state]))
    state, reward, terminated, truncated, _ = render_env.step(action)
    done = terminated or truncated
    steps += 1
    cumulative += reward
    print(
        f"Step {steps}: action={ACTION_NAMES[action]:<5} "
        f"reward={reward:.1f} cumulative={cumulative:.1f}"
    )
    print(render_env.render())
    print()

if cumulative >= 1.0:
    print(f"Reached the goal in {steps} steps with cumulative reward {cumulative:.1f}.")
else:
    print(f"Episode ended in {steps} steps with cumulative reward {cumulative:.1f}.")

render_env.close()


### Step 6: Evaluation summary

**What this section does:** Prints final training statistics and a pass/fail style message based on success rate and greedy episode outcome.

**Why we do it:** On the deterministic 4x4 map, a well-trained agent should reach success rates near 100% and navigate greedily to the goal. Low success suggests needing more episodes or different hyperparameters.


**What to look for in the output**

- Episodes trained: 10,000.
- Final success rate above 80% on a good run.
- Greedy episode cumulative reward: 1.0.
- `Result: policy learned successfully.` when both conditions hold.


In [ ]:
print("Q-learning evaluation summary:")
print(f"  Episodes trained:     {N_EPISODES:,}")
print(f"  Final success rate:   {final_success:.1f}% (last 1000)")
print(f"  Q-table max value:    {Q.max():.4f}")
print(f"  Greedy episode steps: {steps}")
print(f"  Greedy cumulative:    {cumulative:.1f}")
print()
if final_success > 80.0 and cumulative >= 1.0:
    print("Result: policy learned successfully.")
else:
    print("Result: review hyperparameters or episode count if success remains low.")
